In [1]:
"""
qe_format.py - Formats Quantum ESPRESSO input files generated by ASE.

Features:
1. Formats namelist keys (=) to align globally.
2. Removes empty namelist blocks (&IONS /, &CELL /, etc.).
3. Formats card tables (ATOMIC_SPECIES, CELL_PARAMETERS, ATOMIC_POSITIONS)
   into cleanly aligned columns.
4. Relabels species to match custom pymatgen kind names.
"""

from __future__ import annotations

from typing import Dict, Sequence

import re
from ase import Atoms


_ASSIGN_RE = re.compile(r'^([A-Za-z_][\w\(\)]*)\s*=\s*(.*?)(,)?\s*$')

KNOWN_CARDS = (
    "ATOMIC_SPECIES",
    "ATOMIC_POSITIONS",
    "K_POINTS",
    "CELL_PARAMETERS",
    "CONSTRAINTS",
    "OCCUPATIONS",
    "ATOMIC_FORCES",
    "ADDITIONAL_K_POINTS"
)

In [2]:
def build_species_rename_map(
    atoms: Atoms, 
    kind_names: Sequence[str]
) -> Dict[str, str]:
    """Map ASE's auto-generated species labels to desired kind names."""
    magmoms = atoms.get_initial_magnetic_moments()
    seen: dict = {}
    rename_map: Dict[str, str] = {}
    for symbol, magmom, kind in zip(atoms.get_chemical_symbols(), magmoms, kind_names):
        key = (symbol, magmom)
        if key not in seen:
            count_so_far = sum(1 for s, _ in seen if s == symbol)
            ase_label = symbol if count_so_far == 0 else f"{symbol}{count_so_far}"
            seen[key] = ase_label
            rename_map[ase_label] = kind
    return rename_map


def relabel_qe_species(
    filename: str, 
    rename_map: Dict[str, str]
) -> None:
    """Rewrite species labels on ATOMIC_SPECIES and ATOMIC_POSITIONS lines."""
    lines = open(filename).read().splitlines(keepends=True)
    out = []
    in_species_or_positions = False
    for line in lines:
        stripped = line.strip()
        upper = stripped.upper()
        if upper.startswith(("ATOMIC_SPECIES", "ATOMIC_POSITIONS")):
            in_species_or_positions = True
            out.append(line)
            continue
        if upper.startswith(("K_POINTS", "CELL_PARAMETERS", "CONSTRAINTS",
                              "OCCUPATIONS", "ATOMIC_FORCES", "ADDITIONAL_K_POINTS")):
            in_species_or_positions = False
            out.append(line)
            continue
        if in_species_or_positions and stripped:
            parts = line.split(None, 1)
            label = parts[0]
            if label in rename_map:
                new_label = rename_map[label]
                if len(parts) > 1:
                    out.append(f"{new_label:<4}{parts[1]}")
                else:
                    out.append(f"{new_label}{line[len(line.rstrip()):]}")
                continue
        out.append(line)
    with open(filename, "w") as f:
        f.writelines(out)

In [3]:
def format_qe_namelists_centralized(
    text: str, 
    indent: int = 3
) -> str:
    """Formats QE namelists so that all equal signs (=) across ALL blocks

    align to the same global column.
    """
    lines = text.splitlines()
    parsed_blocks = []
    i = 0
    global_max_width = 0

    # PASS 1: Parse all blocks and determine global maximum key length
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        if stripped.startswith('&') and stripped.upper() != '&END':
            block_header = line
            i += 1
            entries = []  # (key, value_or_verbatim_line, has_comma)

            while i < len(lines) and lines[i].strip() != '/' and not lines[i].strip().upper().startswith('&END'):
                s = lines[i].strip()
                m = _ASSIGN_RE.match(s) if s else None
                if m:
                    key, value, comma = m.group(1), m.group(2), m.group(3)
                    entries.append((key, value, comma is not None))
                    # Update global maximum width across all blocks
                    global_max_width = max(global_max_width, len(key))
                elif s:
                    entries.append((None, lines[i], False))  # verbatim passthrough
                i += 1

            closing = lines[i] if i < len(lines) else '/'
            i += 1
            parsed_blocks.append(('block', block_header, entries, closing))
        else:
            parsed_blocks.append(('line', line))
            i += 1

    # PASS 2: Reconstruct output using global_max_width for formatting
    out_lines = []
    for item in parsed_blocks:
        if item[0] == 'line':
            out_lines.append(item[1])
        elif item[0] == 'block':
            _, block_header, entries, closing = item
            out_lines.append(block_header)

            for key, value, has_comma in entries:
                if key is None:
                    out_lines.append(value)
                else:
                    comma = ',' if has_comma else ''
                    out_lines.append(f"{' ' * indent}{key:>{global_max_width}} = {value}{comma}")

            out_lines.append(closing)

    return '\n'.join(out_lines) + ('\n' if text.endswith('\n') else '')


# def format_qe_file_centralized(
#     filename: str, 
#     indent: int = 3
# ) -> None:
#     """In-place version of `format_qe_namelists_centralized` for a file on disk."""
#     with open(filename, 'r') as f:
#         text = f.read()

#     formatted = format_qe_namelists_centralized(text, indent=indent)

#     with open(filename, "w") as f:
#         f.write(formatted)

In [4]:
def _format_table_lines(lines: List[str]) -> List[str]:
    """Format tabular lines (ATOMIC_SPECIES, ATOMIC_POSITIONS, CELL_PARAMETERS)

    into neatly spaced, left-aligned columns.
    """
    if not lines:
        return []

    split_lines = [line.split() for line in lines]
    max_cols = max(len(row) for row in split_lines)

    # Calculate column widths
    col_widths = [0] * max_cols
    for row in split_lines:
        for idx, token in enumerate(row):
            col_widths[idx] = max(col_widths[idx], len(token))

    formatted = []
    for row in split_lines:
        formatted_row = []
        for idx, token in enumerate(row):
            # Left align species, right align numbers
            if idx == 0:
                formatted_row.append(f"{token:<{col_widths[idx]}}")
            else:
                formatted_row.append(f"{token:>{col_widths[idx]}}")
        formatted.append("  ".join(formatted_row))

    return formatted


def format_qe_file_centralized(
    filename: str, 
    indent: int = 3
) -> None:
    """Format QE file in-place: clean empty blocks, align namelists,

    and format card tables into aligned columns.
    """
    with open(filename, 'r') as f:
        lines = f.read().splitlines()

    parsed_blocks = []
    i = 0
    global_max_width = 0

    # PASS 1: Parse content
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        # Handle Namelists (&CONTROL, &SYSTEM, etc.)
        if stripped.startswith('&') and stripped.upper() != '&END':
            block_header = line
            i += 1
            entries = []

            while i < len(lines) and lines[i].strip() != '/' and not lines[i].strip().upper().startswith('&END'):
                s = lines[i].strip()
                m = _ASSIGN_RE.match(s) if s else None
                if m:
                    key, value, comma = m.group(1), m.group(2), m.group(3)
                    entries.append((key, value, comma is not None))
                    global_max_width = max(global_max_width, len(key))
                elif s:
                    entries.append((None, lines[i], False))
                i += 1

            closing = lines[i] if i < len(lines) else '/'
            i += 1

            # Skip empty namelist blocks (e.g., &IONS /)
            if entries:
                parsed_blocks.append(('namelist', block_header, entries, closing))

        # Handle Card sections (ATOMIC_SPECIES, CELL_PARAMETERS, ATOMIC_POSITIONS, etc.)
        elif any(stripped.upper().startswith(card) for card in KNOWN_CARDS):
            card_header = line
            i += 1
            card_data = []

            while i < len(lines):
                next_line = lines[i]
                next_stripped = next_line.strip()
                if (next_stripped.startswith('&') or 
                    any(next_stripped.upper().startswith(c) for c in KNOWN_CARDS)):
                    break
                if next_stripped:
                    card_data.append(next_stripped)
                i += 1

            parsed_blocks.append(('card', card_header, card_data))
        else:
            if stripped:
                parsed_blocks.append(('line', line))
            i += 1

    # PASS 2: Reconstruct output
    out_lines = []
    for item in parsed_blocks:
        kind = item[0]

        if kind == 'line':
            out_lines.append(item[1])

        elif kind == 'namelist':
            _, block_header, entries, closing = item
            out_lines.append(block_header)
            for key, value, has_comma in entries:
                if key is None:
                    out_lines.append(value)
                else:
                    comma = ',' if has_comma else ''
                    out_lines.append(f"{' ' * indent}{key:>{global_max_width}} = {value}{comma}")
            out_lines.append(closing)
            out_lines.append("")  # Blank line separator

        elif kind == 'card':
            _, card_header, card_data = item
            out_lines.append(card_header)
            formatted_card = _format_table_lines(card_data)
            out_lines.extend(formatted_card)
            out_lines.append("")  # Blank line separator

    with open(filename, "w") as f:
        f.write('\n'.join(out_lines).strip() + '\n')